In [1]:
from __future__ import annotations
from typing import Union, Sequence
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


pd.set_option('display.max_colwidth', None)

#### import original datset

In [2]:
# === Load Dataset ===
df = pd.read_csv("../dataset/original-dataset.csv")
print("Shape:", df.shape)
print(df.dtypes)

Shape: (163511, 13)
product_sku_code                 int64
customer                           str
year                             int64
yearweek                         int64
nielsen_total_volume             int64
promotion_indicator              int64
top_brand                          str
flavor_internal                    str
pack_type_internal                 str
pack_size_internal                 str
units_per_package_internal         str
material_medium_description        str
price_per_item                 float64
dtype: object


#### Missing Values Summary

In [3]:
# Missing Values Summary
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage (%)": missing_percent.round(2)
})

display(missing_df)

,Missing_Count,Missing_Percentage (%)
product_sku_code,0,0.00
customer,0,0.00
year,0,0.00
yearweek,0,0.00
nielsen_total_volume,0,0.00
promotion_indicator,0,0.00
top_brand,0,0.00
flavor_internal,0,0.00
pack_type_internal,0,0.00
pack_size_internal,0,0.00


#### Duplicate Rows

In [4]:
dup_mask = df.duplicated(keep=False)
duplicate_rows = df[dup_mask].copy()
print(f"Total duplicate rows (including original occurrences): {len(duplicate_rows)}")
num_unique_duplicate_patterns = duplicate_rows.drop_duplicates().shape[0]
print(f"Number of distinct duplicate rows: {num_unique_duplicate_patterns}")

# display(duplicate_rows)

Total duplicate rows (including original occurrences): 1616
Number of distinct duplicate rows: 392


In [5]:
# Keep first occurrence, remove the rest
df = df.drop_duplicates(keep="first").copy()

print("Original rows:", len(df))
print("After removing duplicates:", len(df))
print("Removed rows:", len(df) - len(df))

Original rows: 162287
After removing duplicates: 162287
Removed rows: 0


#### invalid values

##### Verification methods

In [6]:
CUSTOMER = {
    "L2_ASDA",
    "L2_CRTG",
    "L2_MORRISONS",
    "L2_SAINSBURY'S",
    "L2_TESCO",
}

TOP_BRAND = {
    "APPLETISER",
    "COCA-COLA",
    "COCA-COLA LIGHT / DIET COKE",
    "COCA-COLA ZERO / NO SUGAR",
    "COSTA",
    "DIET OASIS",
    "DR. PEPPER",
    "DR. PEPPER DIET",
    "FANTA",
    "FANTA ZERO / NO SUGAR / DIET / LIGHT",
    "GLACEAU",
    "LILT",
    "LILT DIET",
    "MONSTER",
    "OASIS",
    "POWERADE",
    "REIGN",
    "RELENTLESS",
    "SCHWEPPES",
    "SCHWEPPES DIET",
    "SPRITE",
    "SPRITE ZERO / NO SUGAR / DIET / LIGHT",
}

FLAVOR_INTERNAL = {
    "ACAI & BLUEBERRY & POMEGRANATE",
    "APERITIVO",
    "APPLE",
    "APPLE & KIWI",
    "APPLE & RASPBERRY",
    "ASSORTED",
    "BERRY",
    "BERRY & CARROT & HIBISCUS",
    "BERRY & FRUIT",
    "BLUEBERRY & MOJITO",
    "BROWNIE & CHOCOLATE",
    "CARAMEL",
    "CHERRY",
    "CITRUS",
    "COCONUT & PEACH",
    "COLA",
    "COLA & CHERRY",
    "COLA & CINNAMON",
    "COLA & LEMON",
    "COLA & LIME",
    "COLA & OREO",
    "COLA & VANILLA",
    "CRANBERRY",
    "DRAGONFRUIT",
    "FRUIT",
    "FRUIT & SODA",
    "GINGER",
    "GINGERBREAD",
    "GRAPE",
    "GRAPEFRUIT",
    "GRAPEFRUIT & PINEAPPLE",
    "GUAVA",
    "GUAVA & ORANGE & PASSION FRUIT",
    "KHAOTIC",
    "KIWI",
    "KIWI & STRAWBERRY",
    "LEMON",
    "LEMON & LIME",
    "LEMON & RASPBERRY",
    "MANGO",
    "MELON",
    "MELON & WATERMELON",
    "MIXXD",
    "MOJITO",
    "NOT FLAVOURED",
    "NOT STATED",
    "OAT",
    "ORANGE",
    "PACIFIC PUNCH",
    "PARADISE",
    "PASSION FRUIT",
    "PEACH",
    "PINEAPPLE",
    "PIPELINE PUNCH",
    "RASPBERRY",
    "RIO PUNCH",
    "RIPPER",
    "ROSE",
    "SODA",
    "STRAWBERRY",
    "SUPER DRY",
    "TIRAMISU",
    "TONIC",
    "TONIC & ELDERFLOWER",
    "TONIC & GRAPEFRUIT",
    "TONIC & LEMON",
    "VANILLA",
    "WATERLEMON",
    "WATERMELON",
}

PACK_TYPE_INTERNAL = {
    "CAN",
    "GLASS",
    "PET",
}


In [7]:
def _empty_err(df: pd.DataFrame) -> pd.DataFrame:
    return df.iloc[0:0].assign(
        _rule=pd.Series(dtype="string"),
        _error=pd.Series(dtype="string"),
    )


def _err_df(df: pd.DataFrame, mask: pd.Series, rule: str, msg: str) -> pd.DataFrame:
    if mask is None:
        return _empty_err(df)
    mask = mask.fillna(False)
    if not bool(mask.any()):
        return _empty_err(df)
    out = df.loc[mask].copy()
    out["_rule"] = rule
    out["_error"] = msg
    return out


def validate_product_sku_code(
    df: pd.DataFrame,
    col: str = "product_sku_code",
    min_digits: int = 4,
    max_digits: int = 10,
    key: list = ["product_sku_code","customer","yearweek","promotion_indicator"]
) -> pd.DataFrame:
    raw = df[col]
    errs = []

    # nulls
    errs.append(_err_df(df, raw.isna(), f"{col}.null", "null value"))

    # coerce to numeric
    num = pd.to_numeric(raw, errors="coerce")

    # parse errors (non-null but not numeric)
    errs.append(_err_df(df, raw.notna() & num.isna(), f"{col}.non_numeric", "not a valid number (cannot parse)"))

    # must be integer (no decimals)
    errs.append(_err_df(df, num.notna() & (num % 1 != 0), f"{col}.not_integer", "must be an integer (no decimals)"))

    # numeric rules (safe)
    errs.append(_err_df(df, num.notna() & (num <= 0), f"{col}.non_positive", "must be > 0"))

    # digit length rules (apply only to integer-valued, positive)
    int_num = num.dropna()
    int_num = int_num[int_num % 1 == 0]
    int_num = int_num.astype("int64")

    digits = int_num.abs().astype(str).str.len()
    digit_bad_index = digits.index[(digits < min_digits) | (digits > max_digits)]
    digit_bad_mask = pd.Series(False, index=df.index)
    digit_bad_mask.loc[digit_bad_index] = True
    errs.append(_err_df(df, digit_bad_mask, f"{col}.digit_length", f"digit length must be in [{min_digits},{max_digits}]"))

    # duplicate business key (if columns exist)
    if all(k in df.columns for k in key):
        dup_mask = df.duplicated(subset=key, keep=False)
        errs.append(_err_df(df, dup_mask, f"{col}.duplicate_key", f"duplicate rows on key={key}"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_product_sku_code] error rows: {len(out)}")
    return out


def validate_customer(
    df: pd.DataFrame,
    col: str = "customer",
    allowed: set | None = None,
) -> pd.DataFrame:
    if allowed is None:
        # expects you already defined CUSTOMER globally; fallback if not passed
        allowed = globals().get("CUSTOMER", set())

    raw = df[col]
    errs = []

    s = raw.astype("string").str.strip()

    errs.append(_err_df(df, s.isna(), f"{col}.null", "null value"))
    errs.append(_err_df(df, s.notna() & (s == ""), f"{col}.empty", "empty string"))
    if allowed:
        errs.append(_err_df(df, ~s.isin(allowed) & s.notna(), f"{col}.invalid_value", f"must be one of {sorted(allowed)}"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_customer] error rows: {len(out)}")
    return out


def validate_year(
    df: pd.DataFrame,
    col: str = "year",
    yearweek_col: str = "yearweek",
    min_year: int = 2000,
    max_year: int = 2100,
) -> pd.DataFrame:
    raw = df[col]
    errs = []

    # ---- base checks ----
    errs.append(_err_df(df, raw.isna(), f"{col}.null", "null value"))

    year_num = pd.to_numeric(raw, errors="coerce")
    errs.append(_err_df(df, raw.notna() & year_num.isna(), f"{col}.non_numeric", "not a valid number (cannot parse)"))
    errs.append(_err_df(df, year_num.notna() & (year_num % 1 != 0), f"{col}.not_integer", "must be an integer (no decimals)"))
    errs.append(_err_df(df, year_num.notna() & ~year_num.between(min_year, max_year), f"{col}.out_of_range", f"must be in [{min_year},{max_year}]"))

    # ---- cross-field consistency: year == floor(yearweek/100) ----
    if yearweek_col in df.columns:
        yw_raw = df[yearweek_col]
        yw_num = pd.to_numeric(yw_raw, errors="coerce")

        # only compare where both parse cleanly to integer
        comparable = (
            year_num.notna()
            & yw_num.notna()
            & (year_num % 1 == 0)
            & (yw_num % 1 == 0)
        )

        yw_year_part = (yw_num // 100)
        mismatch = comparable & (year_num != yw_year_part)

        errs.append(_err_df(
            df,
            mismatch,
            f"{col}.mismatch_yearweek",
            f"must equal {yearweek_col}//100 (first 4 digits of {yearweek_col})",
        ))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_year] error rows: {len(out)}")
    return out


def validate_yearweek(
    df: pd.DataFrame,
    col: str = "yearweek",
    year_col: str = "year",
    min_year: int = 2000,
    max_year: int = 2100,
    max_week: int = 52,
) -> pd.DataFrame:
    raw = df[col]
    errs = []

    # ---- base checks ----
    errs.append(_err_df(df, raw.isna(), f"{col}.null", "null value"))

    yw_num = pd.to_numeric(raw, errors="coerce")
    errs.append(_err_df(df, raw.notna() & yw_num.isna(), f"{col}.non_numeric", "not a valid number (cannot parse)"))
    errs.append(_err_df(df, yw_num.notna() & (yw_num % 1 != 0), f"{col}.not_integer", "must be an integer (no decimals)"))

    # 6-digit YYYYWW
    errs.append(_err_df(df, yw_num.notna() & ((yw_num < 100000) | (yw_num > 999999)), f"{col}.digits", "must be 6-digit YYYYWW"))

    # compute parts safely (only when integer)
    safe = yw_num.notna() & (yw_num % 1 == 0)
    safe_yw = yw_num.where(safe)

    year_part = (safe_yw // 100)
    week_part = (safe_yw % 100)

    errs.append(_err_df(df, safe_yw.notna() & ~year_part.between(min_year, max_year), f"{col}.year_part", f"year part must be in [{min_year},{max_year}]"))
    errs.append(_err_df(df, safe_yw.notna() & ~week_part.between(1, max_week), f"{col}.week_part", f"week part must be in [1,{max_week}]"))

    # ---- cross-field consistency: floor(yearweek/100) == year ----
    if year_col in df.columns:
        y_raw = df[year_col]
        y_num = pd.to_numeric(y_raw, errors="coerce")

        comparable = (
            safe_yw.notna()
            & y_num.notna()
            & (y_num % 1 == 0)
        )

        mismatch = comparable & (year_part != y_num)

        errs.append(_err_df(
            df,
            mismatch,
            f"{col}.mismatch_year",
            f"first 4 digits (i.e., {col}//100) must equal {year_col}",
        ))

    out = pd.concat([e for e in errs if not e.empty], axis=0) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_yearweek] error rows: {len(out)}")
    return out


def validate_nielsen_total_volume(
    df: pd.DataFrame,
    col: str = "nielsen_total_volume",
) -> pd.DataFrame:
    raw = df[col]
    errs = []

    errs.append(_err_df(df, raw.isna(), f"{col}.null", "null value"))

    num = pd.to_numeric(raw, errors="coerce")
    errs.append(_err_df(df, raw.notna() & num.isna(), f"{col}.non_numeric", "not a valid number (cannot parse)"))
    errs.append(_err_df(df, num.notna() & (num % 1 != 0), f"{col}.not_integer", "must be an integer (no decimals)"))

    # split errors: volume < 0 and volume == 0
    errs.append(_err_df(df, num.notna() & (num < 0), f"{col}.negative", "value < 0"))
    errs.append(_err_df(df, num.notna() & (num == 0), f"{col}.zero", "value = 0"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_nielsen_total_volume] error rows: {len(out)}")
    return out


def validate_promotion_indicator(
    df: pd.DataFrame,
    col: str = "promotion_indicator",
) -> pd.DataFrame:
    raw = df[col]
    errs = []

    errs.append(_err_df(df, raw.isna(), f"{col}.null", "null value"))

    num = pd.to_numeric(raw, errors="coerce")
    errs.append(_err_df(df, raw.notna() & num.isna(), f"{col}.non_numeric", "not a valid number (cannot parse)"))

    # must be integer
    errs.append(_err_df(df, num.notna() & (num % 1 != 0), f"{col}.not_integer", "must be an integer (no decimals)"))

    # must be 0 or 1
    errs.append(_err_df(df, num.notna() & ~num.isin([0, 1]), f"{col}.invalid_value", "must be 0 or 1"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_promotion_indicator] error rows: {len(out)}")
    return out


def validate_top_brand(
    df: pd.DataFrame,
    col: str = "top_brand",
    allowed: set | None = None,
) -> pd.DataFrame:
    if allowed is None:
        allowed = globals().get("TOP_BRAND", set())

    raw = df[col]
    s = raw.astype("string").str.strip()
    errs = []

    errs.append(_err_df(df, s.isna(), f"{col}.null", "null value"))
    errs.append(_err_df(df, s.notna() & (s == ""), f"{col}.empty", "empty string"))
    if allowed:
        errs.append(_err_df(df, ~s.isin(allowed) & s.notna(), f"{col}.invalid_value", "value not in approved brand list"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_top_brand] error rows: {len(out)}")
    return out


def validate_flavor_internal(
    df: pd.DataFrame,
    col: str = "flavor_internal",
    allowed: set | None = None,
) -> pd.DataFrame:
    if allowed is None:
        allowed = globals().get("FLAVOR_INTERNAL", set())

    raw = df[col]
    s = raw.astype("string").str.strip()
    errs = []

    errs.append(_err_df(df, s.isna(), f"{col}.null", "null value"))
    errs.append(_err_df(df, s.notna() & (s == ""), f"{col}.empty", "empty string"))
    if allowed:
        errs.append(_err_df(df, ~s.isin(allowed) & s.notna(), f"{col}.invalid_value", "value not in approved flavor list"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_flavor_internal] error rows: {len(out)}")
    return out


def validate_pack_type_internal(
    df: pd.DataFrame,
    col: str = "pack_type_internal",
    allowed: set | None = None,
) -> pd.DataFrame:
    if allowed is None:
        allowed = globals().get("PACK_TYPE_INTERNAL", set())

    raw = df[col]
    s = raw.astype("string").str.strip()
    errs = []

    errs.append(_err_df(df, s.isna(), f"{col}.null", "null value"))
    errs.append(_err_df(df, s.notna() & (s == ""), f"{col}.empty", "empty string"))
    if allowed:
        errs.append(_err_df(df, ~s.isin(allowed) & s.notna(), f"{col}.invalid_value", "value not in approved pack type list"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_pack_type_internal] error rows: {len(out)}")
    return out


def validate_pack_size_internal(
    df: pd.DataFrame,
    col: str = "pack_size_internal",
) -> pd.DataFrame:
    raw = df[col]
    errs = []

    # nulls
    errs.append(_err_df(df, raw.isna(), f"{col}.null", "null value"))

    # coerce to numeric safely
    num = pd.to_numeric(raw, errors="coerce")

    # non-numeric values (non-null but cannot parse)
    errs.append(_err_df(df, raw.notna() & num.isna(),
                        f"{col}.non_numeric", "not a valid number (cannot parse)"))

    # must be integer (no decimals)
    errs.append(_err_df(df, num.notna() & (num % 1 != 0),
                        f"{col}.not_integer", "must be an integer (no decimals)"))

    # must be > 0
    errs.append(_err_df(df, num.notna() & (num <= 0),
                        f"{col}.non_positive", "must be > 0"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_pack_size_internal] error rows: {len(out)}")
    return out


def validate_units_per_pack(
    df: pd.DataFrame,
    col: str = "units_per_package_internal",
) -> pd.DataFrame:
    raw = df[col]
    errs = []

    errs.append(_err_df(df, raw.isna(), f"{col}.null", "null value"))

    num = pd.to_numeric(raw, errors="coerce")
    errs.append(_err_df(df, raw.notna() & num.isna(), f"{col}.non_numeric", "not a valid number (cannot parse)"))

    errs.append(_err_df(df, num.notna() & (num <= 0), f"{col}.non_positive", "must be > 0"))
    errs.append(_err_df(df, num.notna() & (num % 1 != 0), f"{col}.not_integer", "must be an integer (no decimals)"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_units_per_pack] error rows: {len(out)}")
    return out


def validate_material_medium_description(
    df: pd.DataFrame,
    col: str = "material_medium_description",
) -> pd.DataFrame:
    raw = df[col]
    errs = []

    s = raw.astype("string").str.strip()

    errs.append(_err_df(df, s.isna(), f"{col}.null", "null value"))
    errs.append(_err_df(df, s.notna() & (s == ""), f"{col}.empty", "empty string"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_material_medium_description] error rows: {len(out)}")
    return out


def validate_price_per_item(
    df: pd.DataFrame,
    col: str = "price_per_item",
    allow_zero: bool = False,
) -> pd.DataFrame:
    raw = df[col]
    errs = []

    errs.append(_err_df(df, raw.isna(), f"{col}.null", "null value"))

    num = pd.to_numeric(raw, errors="coerce")
    errs.append(_err_df(df, raw.notna() & num.isna(), f"{col}.non_numeric", "not a valid number (cannot parse)"))

    if allow_zero:
        errs.append(_err_df(df, num.notna() & (num < 0), f"{col}.negative", "must be >= 0"))
    else:
        errs.append(_err_df(df, num.notna() & (num <= 0), f"{col}.non_positive", "must be > 0"))

    out = pd.concat([e for e in errs if not e.empty], ignore_index=False) \
        if any(not e.empty for e in errs) else _empty_err(df)
    print(f"[validate_price_per_item] error rows: {len(out)}")
    return out


##### Validate

In [8]:
# product_sku_code must be a not null int
err_product_sku_code = validate_product_sku_code(df)

# customer must be a not null string in one of CUSTOMER
err_customer = validate_customer(df)

# year must be a not null 4 digit int
err_year = validate_year(df)

# yearweek must be a not null 6 digit int
err_yearweek = validate_yearweek(df)

# volume must be a not null int with value > 0
err_nielsen_total_volume = validate_nielsen_total_volume(df)

# promotion_indicator must be a not null int either 0 or 1
err_promotion_indicator = validate_promotion_indicator(df)

# top_brand must be a not null string in one of TOP_BRAND
err_top_brand = validate_top_brand(df)

# flavor_internal must be a not null string in one of FLAVOR_INTERNAL
err_flavor_internal = validate_flavor_internal(df)

# pack_type_internal must be a not null string in one of PACK_TYPE_INTERNAL
err_pack_type_internal = validate_pack_type_internal(df)

# pack_size_internal must be a not null int
err_pack_size_internal = validate_pack_size_internal(df)

# units_per_pack must be a not null int
err_units_per_pack = validate_units_per_pack(df)

# material_medium_description must be a not null string
err_material_medium_description = validate_material_medium_description(df)

# validate_price_per_item must be a not null float > 0
err_price_per_item = validate_price_per_item(df)

[validate_product_sku_code] error rows: 380
[validate_customer] error rows: 0
[validate_year] error rows: 2023
[validate_yearweek] error rows: 2023
[validate_nielsen_total_volume] error rows: 18014
[validate_promotion_indicator] error rows: 0
[validate_top_brand] error rows: 0
[validate_flavor_internal] error rows: 0
[validate_pack_type_internal] error rows: 0
[validate_pack_size_internal] error rows: 162287
[validate_units_per_pack] error rows: 162287
[validate_material_medium_description] error rows: 0
[validate_price_per_item] error rows: 65826


In [9]:
PRICE_COL = "price_per_item"
SKU_COL = "product_sku_code"
CUST_COL = "customer"
TIME_COL = "yearweek"
PROMO_COL = "promotion_indicator"
VOL_COL = "nielsen_total_volume"

#### units_per_pack and pack_size_internal datatype error

In [10]:
# we first look at the easiest error: validate_pack_size_internal, validate_units_per_pack
display(err_pack_size_internal["_error"].value_counts(dropna=False))
display(err_units_per_pack["_error"].value_counts(dropna=False))

_error
not a valid number (cannot parse)    162287
Name: count, dtype: int64

_error
not a valid number (cannot parse)    162287
Name: count, dtype: int64

In [11]:
# the output shows that it should be turned from string to int e.g. '500ML' to 500, 'X4' to 4
def convert_pack_size(df, column="pack_size_internal"):
    """
    Convert pack size text (e.g., '500ML') into integer milliliters.
    Assumes all values are in ML units.
    """
    return (
        df[column]
        .str.upper()            # standardize case
        .str.replace("ML", "")  # remove 'ML'
        .str.strip()            # remove spaces if any
        .astype(int)            # convert to integer
    )


def convert_units(df, column="units_per_package_internal"):
    """
    Convert units text (e.g., 'X4') into integer unit count.
    Assumes format always starts with 'X'.
    """
    return (
        df[column]
        .str.upper()            # standardize case
        .str.replace("X", "")   # remove leading 'X'
        .astype(int)            # convert to integer
    )


# we call the above method to solve this datatype error
df["pack_size_internal"] = convert_pack_size(df)
df["units_per_package_internal"] = convert_units(df)

# now we test again, this time there should not be any error left
validate_pack_size_internal(df)
validate_units_per_pack(df)

[validate_pack_size_internal] error rows: 0
[validate_units_per_pack] error rows: 0


,product_sku_code,customer,year,yearweek,nielsen_total_volume,promotion_indicator,top_brand,flavor_internal,pack_type_internal,pack_size_internal,units_per_package_internal,material_medium_description,price_per_item,_rule,_error


#### year and yearweek error

In [12]:
# This is an error where the year and yearweek does not match
display(err_year["_error"].value_counts(dropna=False))
display(err_yearweek["_error"].value_counts(dropna=False))

_error
must equal yearweek//100 (first 4 digits of yearweek)    2023
Name: count, dtype: int64

_error
first 4 digits (i.e., yearweek//100) must equal year    2023
Name: count, dtype: int64

In [13]:
# The previous output shows that only for 202501 and 202401, they did not match the year
err_year.head(3)

,product_sku_code,customer,year,yearweek,nielsen_total_volume,promotion_indicator,top_brand,flavor_internal,pack_type_internal,pack_size_internal,units_per_package_internal,material_medium_description,price_per_item,_rule,_error
49264,640044,L2_TESCO,2024,202501,340,1,COSTA,NOT STATED,PET,750ML,X1,750MLNRP X6 COSTA LATTE,NaN,year.mismatch_yearweek,must equal yearweek//100 (first 4 digits of yearweek)
49318,643015,L2_TESCO,2024,202501,39,0,REIGN,PEACH,CAN,355ML,X1,355MLCAN X12 REIGN STORM PCH NCTR,NaN,year.mismatch_yearweek,must equal yearweek//100 (first 4 digits of yearweek)
49337,414885,L2_SAINSBURY'S,2024,202501,8418,0,FANTA ZERO / NO SUGAR / DIET / LIGHT,ORANGE,CAN,330ML,X8,330MLCAN 3X8P FTA OR ZRO,NaN,year.mismatch_yearweek,must equal yearweek//100 (first 4 digits of yearweek)


In [14]:
# The previous output shows that all the rows in 202501 have the year 2024, and all the rows in 202401 has 2023
# This is wrong because 202301 behaves normal which have the year of 2023 not 2022, in this case, we simply change the year for these rows to the correct ones
df.loc[df["yearweek"] == 202401, "year"] = 2024
df.loc[df["yearweek"] == 202501, "year"] = 2025

# now we test again, this time there should not be any error left
validate_year(df)
validate_yearweek(df)


[validate_year] error rows: 0
[validate_yearweek] error rows: 0


,product_sku_code,customer,year,yearweek,nielsen_total_volume,promotion_indicator,top_brand,flavor_internal,pack_type_internal,pack_size_internal,units_per_package_internal,material_medium_description,price_per_item,_rule,_error


#### sku reapeat error

In [15]:
# this is this the error of sku from the original df:
display(err_product_sku_code["_error"].value_counts(dropna=False))

_error
duplicate rows on key=['product_sku_code', 'customer', 'yearweek', 'promotion_indicator']    380
Name: count, dtype: int64

In [16]:
# err_product_sku_code.head()
KEY = [SKU_COL, CUST_COL, TIME_COL, PROMO_COL]    # combindation key

err_unique_keys = (
  err_product_sku_code[KEY]
    .drop_duplicates()
    .sort_values(KEY)
    .reset_index(drop=True)
)

print(f"duplicate groups (unique keys repeated): {len(err_unique_keys)}")

duplicate groups (unique keys repeated): 190


In [17]:
# this is an issue where there are multiple rows with the same 'product_sku_code', 'customer', 'yearweek', 'promotion_indicator'
# this usually is not correct since one sku should have 1 combination of 'product_sku_code', 'customer', 'yearweek', 'promotion_indicator' or even without 'promotion_indicator'
# in the previous output, we see that for all the error rows only the price col differs, we can futher verify calling the validate function again with our own parameter
test = validate_product_sku_code(df, key = [
    "product_sku_code",
    "customer",
    "year",
    "yearweek",
    "promotion_indicator",
    "top_brand",
    "flavor_internal",
    "pack_type_internal",
    "pack_size_internal",
    "units_per_package_internal",
    "material_medium_description"
    ]
)

# this output the same number as above, which means indeed, for all the error redundant rows, only the price differs

[validate_product_sku_code] error rows: 380


In [18]:
# we have several step for cleaning this data
# we first delete all the duplicate rows with 0 values
dup_idx = err_product_sku_code.index
dup_idx = dup_idx.intersection(df.index)

# only drop price==0 within those duplicate rows
dup_prices = df.loc[dup_idx, PRICE_COL]
drop_idx = dup_prices[dup_prices.eq(0)].index

before = len(df)
df = df.drop(index=drop_idx).copy()
after = len(df)

print(f"Duplicate rows flagged: {len(dup_idx)}")
print(f"Dropped (price==0 within duplicates): {len(drop_idx)}")
print(f"Rows before: {before}")
print(f"Rows after: {after}")

Duplicate rows flagged: 380
Dropped (price==0 within duplicates): 159
Rows before: 162287
Rows after: 162128


In [19]:
# we now validate again to update the new repeated error rows
err_product_sku_code = validate_product_sku_code(df)
# display(err_product_sku_code)

err_unique_keys = (
  err_product_sku_code[KEY]
    .drop_duplicates()
    .sort_values(KEY)
    .reset_index(drop=True)
)

print(f"duplicate groups (unique keys repeated): {len(err_unique_keys)}")

[validate_product_sku_code] error rows: 62
duplicate groups (unique keys repeated): 31


In [20]:
# compute ALL-TIME median price per (sku + customer + promo), ignoring 0 and NaN, without modifying df

PRICE_COL = "price_per_item"
GRP = [SKU_COL, CUST_COL, PROMO_COL]  # same sku + customer + promo only

price_num = pd.to_numeric(df[PRICE_COL], errors="coerce")
valid_price = price_num.where(price_num > 0)  # <=0 -> NaN, so median ignores them

median_by_grp_alltime = (
  df.assign(_price_valid=valid_price)
    .groupby(GRP, dropna=False)["_price_valid"]
    .median()
    .reset_index(name="hist_median_price")
)

# attach the historical median to each duplicate key combo
err_unique_keys_with_hist_median = err_unique_keys.merge(
  median_by_grp_alltime,
  on=GRP,
  how="left"
)

display(err_unique_keys_with_hist_median.head(50))

,product_sku_code,customer,yearweek,promotion_indicator,hist_median_price
0,429213,L2_SAINSBURY'S,202419,0,1.7733
1,429213,L2_SAINSBURY'S,202421,0,1.7733
2,429213,L2_SAINSBURY'S,202426,0,1.7733
3,429213,L2_SAINSBURY'S,202427,0,1.7733
4,429213,L2_SAINSBURY'S,202433,0,1.7733
5,429213,L2_SAINSBURY'S,202435,0,1.7733
6,429213,L2_SAINSBURY'S,202438,0,1.7733
7,429213,L2_SAINSBURY'S,202441,0,1.7733
8,429213,L2_SAINSBURY'S,202443,0,1.7733
9,429213,L2_SAINSBURY'S,202445,0,1.7733


In [21]:
# replace the original df by choosing the rows that is the closest to the median for each duplicate combination rows 
PRICE_COL = "price_per_item"
GRP = [SKU_COL, CUST_COL, PROMO_COL]  # sku + customer + promo

# 1) Find all rows in df whose keys are in err_unique_keys (only process these duplicate keys)
df_key_idx = pd.MultiIndex.from_frame(df[KEY])
target_key_idx = pd.MultiIndex.from_frame(err_unique_keys[KEY])
is_target_dup = df_key_idx.isin(target_key_idx)

cand = df.loc[is_target_dup, KEY + [PRICE_COL]].copy()
cand["_row_id"] = cand.index  # store original df row index (used for final drop)

# 2) Convert price to numeric + ignore 0/NaN (do not modify df; only operate in cand)
cand[PRICE_COL] = pd.to_numeric(cand[PRICE_COL], errors="coerce")
cand["_price_valid"] = cand[PRICE_COL].where(cand[PRICE_COL] > 0)

# 3) Merge the all-time median onto candidate rows (grouped by GRP)
cand = cand.merge(
  median_by_grp_alltime,  # columns: GRP + ["hist_median_price"]
  on=GRP,
  how="left"
)

# 4) Compute distance dist, and for each KEY keep the row with the smallest dist
cand["dist"] = (cand["_price_valid"] - cand["hist_median_price"]).abs().fillna(np.inf)

idxmin = cand.groupby(KEY, dropna=False)["dist"].idxmin()
keep_row_ids = cand.loc[idxmin, "_row_id"]

# 5) Drop the other duplicate rows (only drop these duplicates; do not affect other rows)
drop_row_ids = cand.loc[~cand["_row_id"].isin(keep_row_ids), "_row_id"]

before = len(df)
df = df.drop(index=drop_row_ids).copy()
after = len(df)

print("Duplicate key combos:", len(err_unique_keys))
print("Rows removed (kept 1 per key):", before - after)

# (optional) Inspect the final kept rows (includes price / historical median / dist)
chosen = cand[cand["_row_id"].isin(keep_row_ids)][KEY + [PRICE_COL, "hist_median_price", "dist"]].sort_values(KEY)
# display(chosen.head(50))

Duplicate key combos: 31
Rows removed (kept 1 per key): 31


In [22]:
# we now test agian, it should not return any error
err_product_sku_code = validate_product_sku_code(df)

[validate_product_sku_code] error rows: 0


#### price and vol error

In [23]:
# we now test corrilation between invalid volumn and invalid price (<=0)
mask_volume_invalid = df[VOL_COL].isna() | df[VOL_COL].le(0)
mask_price_invalid  = df[PRICE_COL].isna() | df[PRICE_COL].le(0)

volume_invalid = df[mask_volume_invalid]
price_invalid = df[mask_price_invalid]
both_invalid = df[mask_volume_invalid & mask_price_invalid]

print(f"rows with invalid volume: {int(mask_volume_invalid.sum())}")
print(f"rows with invalid price: {int(mask_price_invalid.sum())}")
print(f"rows with invalid volume AND invalid price: {len(both_invalid)}")

# share of volume-invalid rows that also have invalid price
print(f"price-invalid rate within volume-invalid rows: {len(both_invalid) / max(len(volume_invalid), 1):.4%}")

# share of price-invalid rows that also have invalid volume
print(f"volume-invalid rate within price-invalid rows: {len(both_invalid) / max(len(price_invalid), 1):.4%}")

rows with invalid volume: 17936
rows with invalid price: 65667
rows with invalid volume AND invalid price: 15601
price-invalid rate within volume-invalid rows: 86.9815%
volume-invalid rate within price-invalid rows: 23.7577%


In [24]:
# we see that there is high corrilation between invalid volumn and invalid price
# In other words, there is a mojority of rows with invalid price and invalid volume in both all invalid price and volume
# therefore, we first remove all the rows with invalid price AND invalid volume
mask_volume_invalid = df[VOL_COL].isna() | df[VOL_COL].le(0)
mask_price_invalid  = df[PRICE_COL].isna() | df[PRICE_COL].le(0)

mask_drop = mask_volume_invalid & mask_price_invalid

before = len(df)
df = df[~mask_drop].copy()
after = len(df)

print("Rows before:", before)
print("Rows removed (invalid volume(==0) & invalid price):", before - after)
print("Rows after:", after)

Rows before: 162097
Rows removed (invalid volume(==0) & invalid price): 15601
Rows after: 146496


#### price_per_item error

In [25]:
# we now look at the price_per_item error
# original df errors:
display(err_price_per_item["_error"].value_counts(dropna=False))

# the updated df errors:
err_price_per_item = validate_price_per_item(df)
display(err_price_per_item["_error"].value_counts(dropna=False))

_error
null value     56040
must be > 0     9786
Name: count, dtype: int64

[validate_price_per_item] error rows: 50066


_error
null value     49950
must be > 0      116
Name: count, dtype: int64

##### Method 1

In [26]:
def get_invalid(
  df: pd.DataFrame,
  target_col: str,
  group_cols: Union[str, Sequence[str]],
  *,
  coerce_numeric: bool = True
) -> pd.DataFrame:
  """
  Compute rate of (NaN OR <= 0) in `target_col`, grouped by one or multiple columns.

  Params
  ------
  df : pd.DataFrame
  target_col : str
    Column to measure invalids on.
  group_cols : str | Sequence[str]
    Grouping column(s), e.g. "product_sku_code" or ["yearweek", "customer"].
  coerce_numeric : bool
    If True, coerces target_col to numeric; non-parsable values become NaN (treated as invalid).

  Returns
  -------
  pd.DataFrame
    Index: group_cols (or MultiIndex if multiple)
    Columns: row_count, invalid_count, invalid_rate
    Sorted by invalid_rate, invalid_count, row_count (descending).
  """
  if isinstance(group_cols, str):
    group_cols_list = [group_cols]
  else:
    group_cols_list = list(group_cols)

  missing = [c for c in [target_col, *group_cols_list] if c not in df.columns]
  if missing:
    raise KeyError(f"Missing column(s) in df: {missing}")

  s = df[target_col]
  if coerce_numeric:
    s = pd.to_numeric(s, errors="coerce")

  is_invalid = s.isna() | (s <= 0)

  out = (
    df.assign(_is_target_invalid=is_invalid)
      .groupby(group_cols_list, dropna=False)
      .agg(
        row_count=(target_col, "size"),
        invalid_count=("_is_target_invalid", "sum"),
      )
      .assign(invalid_rate=lambda x: x["invalid_count"] / x["row_count"])
      .sort_values(["invalid_rate", "invalid_count", "row_count"], ascending=False)
  )

  return out

In [27]:
# invalid rate by sku
price_invalid_sku = get_invalid(df, PRICE_COL, SKU_COL)
display(price_invalid_sku.head(20))

# how many SKUs have invalid_rate == 1.0
price_invalid_1_sku = price_invalid_sku[price_invalid_sku["invalid_rate"] == 1.0]

# display(price_invalid_1_sku)
print(f"SKUs with price invalid_rate == 1.0: {price_invalid_1_sku.shape[0]} / {price_invalid_sku.shape[0]}")

# (optional) how many rows are covered by those SKUs
rows_covered = price_invalid_1_sku["row_count"].sum()
print(f"Rows covered by SKUs with price invalid_rate == 1.0: {int(rows_covered)} ({rows_covered / len(df):.4%} of all rows)")

,row_count,invalid_count,invalid_rate
product_sku_code,,,
217852,780,780,1.0
254991,780,780,1.0
264471,780,780,1.0
407189,780,780,1.0
408656,780,780,1.0
408700,780,780,1.0
426513,780,780,1.0
429137,780,780,1.0
419293,748,748,1.0


SKUs with price invalid_rate == 1.0: 190 / 595
Rows covered by SKUs with price invalid_rate == 1.0: 41604 (28.3994% of all rows)


In [28]:
# invalid rate of customer
price_invalid_customer = get_invalid(df, PRICE_COL, CUST_COL)

# display(price_invalid_customer)

In [29]:
# invalid rate by yearweek
price_invalid_yearweek = get_invalid(df, PRICE_COL, TIME_COL)
display(price_invalid_yearweek.head(15))

# how many yearweek have invalid_rate == 1.0
price_invalid_1_yearweek = price_invalid_yearweek[price_invalid_yearweek["invalid_rate"] == 1.0]

# display(price_invalid_1_yearweek)
print(f"yearweek with price invalid_rate == 1.0: {price_invalid_1_yearweek.shape[0]} / {price_invalid_yearweek.shape[0]}")

# (optional) how many rows are covered by those SKUs
rows_covered = price_invalid_1_yearweek["row_count"].sum()
print(f"Rows covered by yearweek with price invalid_rate == 1.0: {int(rows_covered)} ({rows_covered / len(df):.4%} of all rows)")

,row_count,invalid_count,invalid_rate
yearweek,,,
202244,909,909,1.000000
202501,904,904,1.000000
202401,882,882,1.000000
202245,869,869,1.000000
202251,855,855,1.000000
202248,854,854,1.000000
202246,852,852,1.000000
202250,848,848,1.000000
202249,844,844,1.000000


yearweek with price invalid_rate == 1.0: 11 / 156
Rows covered by yearweek with price invalid_rate == 1.0: 9480 (6.4712% of all rows)


In [30]:
# invalid rate by promo_indicator
price_invalid_promo = get_invalid(df, PRICE_COL, PROMO_COL)

# display(price_invalid_promo)

In [31]:
# Rule 1: price_invalid_rate > 0.9
# Rule 2: price_invalid_rate > 0.5 AND price_valid_count < 12
# Rule 3: n_unique_valid_prices < 3   (use rounded prices to avoid float noise)
# Rule 5: (p99 / p1) > 10             (unit-mismatch / extreme scale heuristic)

RULE1_T = 0.9
RULE2_T = 0.5
RULE2_MIN_VALID = 12

RULE3_MIN_UNIQUE = 3
RULE5_RATIO_MAX = 10
PRICE_DECIMALS = 2  # round to 2dp before counting unique prices

PRICE_COL = PRICE_COL
SKU_COL = SKU_COL

# ---- base stats from your invalid table ----
price_sku_stats = price_invalid_sku.copy()
price_sku_stats["valid_count"] = price_sku_stats["row_count"] - price_sku_stats["invalid_count"]

# ---- compute Rule 3 & Rule 5 features on the current df (do NOT modify df) ----
price_num = pd.to_numeric(df[PRICE_COL], errors="coerce")
price_valid = price_num.where(price_num > 0)

price_valid_rounded = price_valid.round(PRICE_DECIMALS)

unique_valid_prices = (
  df.assign(_p=price_valid_rounded)
    .groupby(SKU_COL, dropna=False)["_p"]
    .nunique(dropna=True)
    .rename("unique_valid_prices")
)

p1 = (
  df.assign(_p=price_valid)
    .groupby(SKU_COL, dropna=False)["_p"]
    .quantile(0.01)
    .rename("p01")
)
p99 = (
  df.assign(_p=price_valid)
    .groupby(SKU_COL, dropna=False)["_p"]
    .quantile(0.99)
    .rename("p99")
)

extra = pd.concat([unique_valid_prices, p1, p99], axis=1)
extra["p99_p01_ratio"] = extra["p99"] / extra["p01"]

# merge into sku stats
price_sku_stats = price_sku_stats.join(extra, how="left")

# ---- rules ----
rule1_mask = price_sku_stats["invalid_rate"] > RULE1_T
rule2_mask = (price_sku_stats["invalid_rate"] > RULE2_T) & (price_sku_stats["valid_count"] < RULE2_MIN_VALID)

rule3_mask = price_sku_stats["unique_valid_prices"].fillna(0) < RULE3_MIN_UNIQUE

# avoid dividing by 0 / NaN: only apply ratio rule when p01 is valid and > 0
rule5_mask = (
  price_sku_stats["p01"].notna()
  & price_sku_stats["p01"].gt(0)
  & price_sku_stats["p99_p01_ratio"].gt(RULE5_RATIO_MAX)
)

sku_drop = price_sku_stats.index[rule1_mask | rule2_mask | rule3_mask | rule5_mask]
# sku_drop = price_sku_stats.index[rule1_mask | rule2_mask | rule3_mask]

print(f"SKUs to drop (Rule1/2/3/5): {len(sku_drop)} / {price_sku_stats.shape[0]}")

before = len(df)
df = df[~df[SKU_COL].isin(sku_drop)].copy()
after = len(df)

print("Rows before:", before)
print("Rows removed (Rule1/2/3/5 SKUs):", before - after)
print("Rows after:", after)

# ---- report (why dropped) ----
dropped_report = price_sku_stats.loc[sku_drop].copy()
dropped_report["drop_reason"] = (
  np.select(
    [rule1_mask.loc[sku_drop], rule2_mask.loc[sku_drop], rule3_mask.loc[sku_drop], rule5_mask.loc[sku_drop]],
    ["rule1_invalid_rate>0.9", "rule2_invalid_rate>0.5_and_valid<12", "rule3_unique_prices<3", "rule5_p99_p01_ratio>10"],
    default="multiple_rules"
  )
)
display(dropped_report.sort_values(["invalid_rate", "valid_count"], ascending=[False, True]))
# display(dropped_report.sort_values(["drop_reason"], ascending=False).head(30))

SKUs to drop (Rule1/2/3/5): 213 / 595
Rows before: 146496
Rows removed (Rule1/2/3/5 SKUs): 44745
Rows after: 101751


,row_count,invalid_count,invalid_rate,valid_count,unique_valid_prices,p01,p99,p99_p01_ratio,drop_reason
product_sku_code,,,,,,,,,
217852,780,780,1.0,0,0,NaN,NaN,NaN,rule1_invalid_rate>0.9
254991,780,780,1.0,0,0,NaN,NaN,NaN,rule1_invalid_rate>0.9
264471,780,780,1.0,0,0,NaN,NaN,NaN,rule1_invalid_rate>0.9
407189,780,780,1.0,0,0,NaN,NaN,NaN,rule1_invalid_rate>0.9
408656,780,780,1.0,0,0,NaN,NaN,NaN,rule1_invalid_rate>0.9
...,...,...,...,...,...,...,...,...,...
640728,2,0,0.0,2,1,6.000000,6.000000,1.000000,rule3_unique_prices<3
644697,4,0,0.0,4,2,1.550760,1.556719,1.003843,rule3_unique_prices<3
414166,5,0,0.0,5,2,0.560000,2.000000,3.571429,rule3_unique_prices<3


In [32]:
# we then test again the invalid rate of yearweek after we drop the sku rows
# invalid rate by yearweek
price_invalid_yearweek = get_invalid(df, PRICE_COL, TIME_COL)
display(price_invalid_yearweek.head(15))

# how many yearweek have invalid_rate == 1.0
price_invalid_1_yearweek = price_invalid_yearweek[price_invalid_yearweek["invalid_rate"] == 1.0]

# display(price_invalid_1_sku)
print(f"yearweek with price invalid_rate == 1.0: {price_invalid_1_yearweek.shape[0]} / {price_invalid_yearweek.shape[0]}")

# (optional) how many rows are covered by those SKUs
rows_covered = price_invalid_1_yearweek["row_count"].sum()
print(f"Rows covered by yearweek with price invalid_rate == 1.0: {int(rows_covered)} ({rows_covered / len(df):.4%} of all rows)")

,row_count,invalid_count,invalid_rate
yearweek,,,
202244,647,647,1.000000
202401,619,619,1.000000
202501,617,617,1.000000
202245,614,614,1.000000
202246,612,612,1.000000
202248,610,610,1.000000
202249,600,600,1.000000
202251,598,598,1.000000
202250,597,597,1.000000


yearweek with price invalid_rate == 1.0: 11 / 156
Rows covered by yearweek with price invalid_rate == 1.0: 6697 (6.5818% of all rows)


In [33]:
# after dropping the invalid sku, “partial-null weeks” basically disappear, the largest null rate that is not 1 dropped from 0.33 to 0.01
# however there are still yearweek_invalid_rate = 1
# Rule 1: yearweek_invalid_rate > 0.5 (in this case it is the same as delete all yearweek rows with yearweek_invalid_rate = 1)
RULE1_YEARWEEK_T = 0.5

price_yearweek_stats = price_invalid_yearweek.copy()
rule1_mask = price_yearweek_stats["invalid_rate"] > RULE1_YEARWEEK_T
yearweek_drop = price_yearweek_stats.index[rule1_mask]

print(f"yearweek to drop: {len(yearweek_drop)} / {price_yearweek_stats.shape[0]}")

before = len(df)
df = df[~df[TIME_COL].isin(yearweek_drop)].copy()  # <-- fix: filter by TIME_COL
after = len(df)

print("Rows before:", before)
print("Rows removed (yearweek rule1):", before - after)
print("Rows after:", after)

# optional report
dropped_report = price_yearweek_stats.loc[yearweek_drop].copy()
dropped_report["drop_reason"] = "rule1_yearweek_invalid_rate>0.5"
display(dropped_report)

yearweek to drop: 11 / 156
Rows before: 101751
Rows removed (yearweek rule1): 6697
Rows after: 95054


,row_count,invalid_count,invalid_rate,drop_reason
yearweek,,,,
202244,647,647,1.0,rule1_yearweek_invalid_rate>0.5
202401,619,619,1.0,rule1_yearweek_invalid_rate>0.5
202501,617,617,1.0,rule1_yearweek_invalid_rate>0.5
202245,614,614,1.0,rule1_yearweek_invalid_rate>0.5
202246,612,612,1.0,rule1_yearweek_invalid_rate>0.5
202248,610,610,1.0,rule1_yearweek_invalid_rate>0.5
202249,600,600,1.0,rule1_yearweek_invalid_rate>0.5
202251,598,598,1.0,rule1_yearweek_invalid_rate>0.5
202250,597,597,1.0,rule1_yearweek_invalid_rate>0.5


In [34]:
# now we test again, this time the error rows should reduced
err_price_per_item_new = validate_price_per_item(df)
display(err_price_per_item_new["_error"].value_counts(dropna=False))

[validate_price_per_item] error rows: 755


_error
null value     645
must be > 0    110
Name: count, dtype: int64

In [35]:
# Now we try to impute the above rows as they are not structural data missing
df = df.copy()

# --- audit: keep original price (treat <=0 as invalid) ---
df["_price_before_impute"] = pd.to_numeric(df[PRICE_COL], errors="coerce")
df.loc[df["_price_before_impute"] <= 0, "_price_before_impute"] = np.nan

# --- normalize: 0 and negative are invalid -> NaN (for imputation) ---
df[PRICE_COL] = pd.to_numeric(df[PRICE_COL], errors="coerce")
df.loc[df[PRICE_COL] <= 0, PRICE_COL] = np.nan

# 1) forward-fill within sku+customer+promo over time
df = df.sort_values([SKU_COL, CUST_COL, PROMO_COL, TIME_COL]).copy()
df[PRICE_COL] = (
  df.groupby([SKU_COL, CUST_COL, PROMO_COL], dropna=False)[PRICE_COL]
    .ffill()
)

# 2) fallback: median within sku+customer+promo
mask = df[PRICE_COL].isna()
med_grp = df.groupby([SKU_COL, CUST_COL, PROMO_COL], dropna=False)[PRICE_COL].transform("median")
df.loc[mask, PRICE_COL] = df.loc[mask, PRICE_COL].fillna(med_grp[mask])

# 3) fallback: median within sku
mask = df[PRICE_COL].isna()
med_sku = df.groupby(SKU_COL, dropna=False)[PRICE_COL].transform("median")
df.loc[mask, PRICE_COL] = df.loc[mask, PRICE_COL].fillna(med_sku[mask])

# 4) final fallback: global median
mask = df[PRICE_COL].isna()
df.loc[mask, PRICE_COL] = df.loc[mask, PRICE_COL].fillna(df[PRICE_COL].median())

# --- print all filled values (was invalid before, now filled) ---
filled_mask = df["_price_before_impute"].isna() & df[PRICE_COL].notna()
print("Filled rows:", int(filled_mask.sum()))

filled_rows = (
  df.loc[filled_mask, [SKU_COL, CUST_COL, PROMO_COL, TIME_COL, "_price_before_impute", PRICE_COL]]
    .sort_values([SKU_COL, CUST_COL, PROMO_COL, TIME_COL])
)

# drop audit column so it doesn't leak into downstream cells / saved CSV
df = df.drop(columns=["_price_before_impute"])

display(filled_rows)

Filled rows: 755


,product_sku_code,customer,promotion_indicator,yearweek,_price_before_impute,price_per_item
153046,233383,L2_ASDA,0,202330,NaN,1.3000
129475,233383,L2_ASDA,0,202332,NaN,3.0000
132442,233383,L2_ASDA,0,202336,NaN,10.0000
137253,233383,L2_ASDA,0,202339,NaN,8.2500
104064,233383,L2_ASDA,0,202340,NaN,8.2500
...,...,...,...,...,...,...
54299,641548,L2_CRTG,1,202410,NaN,1.5563
57295,641548,L2_CRTG,1,202411,NaN,1.5563
98759,641548,L2_CRTG,1,202412,NaN,1.5563
93679,641548,L2_CRTG,1,202413,NaN,1.5563


In [36]:
# now we validate again, this time there should have no error left
err_price = validate_price_per_item(df)

[validate_price_per_item] error rows: 0


##### Method 2 Delete all the invalid rows

In [37]:
err_price_per_item = validate_price_per_item(df)

before_len = len(df)

bad_idx = err_price_per_item.index.unique()
df = df.drop(index=bad_idx)

after_len = len(df)
dropped = before_len - after_len

print(f"Before df length: {before_len}")
print(f"After df length:  {after_len}")
print(f"Rows dropped:     {dropped}")

[validate_price_per_item] error rows: 0
Before df length: 95054
After df length:  95054
Rows dropped:     0


#### nielsen_total_volume error

In [38]:
# I dont know whether a volumn = 0 is ok or not, I assume volumn = 0 is an invalid value, but this can be changed later if needed
# this is the original df volume error
display(err_nielsen_total_volume["_error"].value_counts(dropna=False))

# this is the latest df volume error after all the changes been made above:
err_nielsen_total_volume = validate_nielsen_total_volume(df)
display(err_nielsen_total_volume["_error"].value_counts(dropna=False))

_error
value = 0    17953
value < 0       61
Name: count, dtype: int64

[validate_nielsen_total_volume] error rows: 2266


_error
value = 0    2240
value < 0      26
Name: count, dtype: int64

##### Method 1

In [39]:
# first of all, we see that there are still some rows with value < 0
err_vol_neg = err_nielsen_total_volume[err_nielsen_total_volume["_error"].eq("value < 0")]
# display(err_vol_neg)

In [40]:
# this is not acceptable, so we remove them
mask_vol_neg = df[VOL_COL].lt(0)

# drop all rows belonging to those SKUs
before = len(df)
df = df[~mask_vol_neg].copy()
after = len(df)

print("Rows before:", before)
print("Rows removed (volume < 0):", before - after)
print("Rows after:", after)

Rows before: 95054
Rows removed (volume < 0): 26
Rows after: 95028


In [41]:
# now we left only the volume == 0, this might be acceptable in reality, so we will investigate this further
# invalid rate by sku
volume_invalid_sku = get_invalid(df, VOL_COL, SKU_COL)
display(volume_invalid_sku.head(5))

# how many SKUs have invalid_rate == 1.0
volume_invalid_1_sku = volume_invalid_sku[volume_invalid_sku["invalid_rate"] == 1.0]

# display(volume_invalid_1_sku)
print(f"SKUs with volume invalid_rate == 1.0: {volume_invalid_1_sku.shape[0]} / {volume_invalid_sku.shape[0]}")

# (optional) how many rows are covered by those SKUs
rows_covered = volume_invalid_1_sku["row_count"].sum()
print(f"Rows covered by SKUs with volume invalid_rate == 1.0: {int(rows_covered)} ({rows_covered / len(df):.4%} of all rows)")

,row_count,invalid_count,invalid_rate
product_sku_code,,,
640007,18,16,0.888889
429213,142,99,0.697183
418336,67,38,0.567164
203475,29,15,0.517241
284918,32,16,0.500000


SKUs with volume invalid_rate == 1.0: 0 / 382
Rows covered by SKUs with volume invalid_rate == 1.0: 0 (0.0000% of all rows)


In [42]:
# invalid rate of customer
volume_invalid_customer = get_invalid(df, VOL_COL, CUST_COL)

display(volume_invalid_customer)

,row_count,invalid_count,invalid_rate
customer,,,
L2_CRTG,13664,431,0.031543
L2_SAINSBURY'S,18162,486,0.026759
L2_MORRISONS,22448,593,0.026417
L2_TESCO,19821,497,0.025074
L2_ASDA,20933,233,0.011131


In [43]:
# invalid rate by yearweek
volume_invalid_yearweek = get_invalid(df, VOL_COL, TIME_COL)
display(volume_invalid_yearweek.head(10))

# how many yearweek have invalid_rate == 1.0
volume_invalid_1_yearweek = volume_invalid_yearweek[volume_invalid_yearweek["invalid_rate"] == 1.0]

# display(volume_invalid_1_yearweek)
print(f"yearweek with volume invalid_rate == 1.0: {volume_invalid_1_yearweek.shape[0]} / {volume_invalid_yearweek.shape[0]}")

# (optional) how many rows are covered by those SKUs
rows_covered = volume_invalid_1_yearweek["row_count"].sum()
print(f"Rows covered by yearweek with volume invalid_rate == 1.0: {int(rows_covered)} ({rows_covered / len(df):.4%} of all rows)")

,row_count,invalid_count,invalid_rate
yearweek,,,
202527,695,30,0.043165
202528,684,29,0.042398
202535,664,28,0.042169
202430,684,28,0.040936
202438,670,26,0.038806
202434,652,25,0.038344
202522,683,26,0.038067
202532,665,25,0.037594
202435,667,25,0.037481


yearweek with volume invalid_rate == 1.0: 0 / 145
Rows covered by yearweek with volume invalid_rate == 1.0: 0 (0.0000% of all rows)


In [44]:
# invalid rate by promo_indicator
volume_invalid_promo = get_invalid(df, VOL_COL, PROMO_COL)

display(volume_invalid_promo)

,row_count,invalid_count,invalid_rate
promotion_indicator,,,
0,41883,1958,0.046749
1,53145,282,0.005306


In [45]:
# Since you already removed NaN and <0, "invalid" here effectively means volume == 0.

POS_WEEKS_MIN = 8          # Rule 1: keep only SKUs with enough weeks where volume > 0
ZERO_RATE_MAX = 0.2       # Rule 2: drop SKUs with too many zero-volume rows

volume_sku_stats = volume_invalid_sku.copy()

# invalid_count ~ zero_count, so positive_weeks = total - zero
volume_sku_stats["positive_weeks"] = volume_sku_stats["row_count"] - volume_sku_stats["invalid_count"]
volume_sku_stats["zero_rate"] = volume_sku_stats["invalid_rate"]

rule1_mask = volume_sku_stats["positive_weeks"] < POS_WEEKS_MIN
rule2_mask = volume_sku_stats["zero_rate"] > ZERO_RATE_MAX

# AND (only drop if both rules are true)
sku_drop_vol = volume_sku_stats.index[rule1_mask & rule2_mask]

print(f"SKUs to drop (Rule1 AND Rule2): {len(sku_drop_vol)} / {volume_sku_stats.shape[0]}")

before = len(df)
df = df[~df[SKU_COL].isin(sku_drop_vol)].copy()
after = len(df)

print("Rows before:", before)
print("Rows removed (volume Rule1&2 SKUs):", before - after)
print("Rows after:", after)

# report
dropped_report_vol = volume_sku_stats.loc[sku_drop_vol].copy()
dropped_report_vol["drop_reason"] = f"rule1_positive_weeks<{POS_WEEKS_MIN} AND rule2_zero_rate>{ZERO_RATE_MAX}"
display(dropped_report_vol.sort_values(["zero_rate", "positive_weeks"], ascending=[False, True]).head(50))

SKUs to drop (Rule1 AND Rule2): 3 / 382
Rows before: 95028
Rows removed (volume Rule1&2 SKUs): 32
Rows after: 94996


,row_count,invalid_count,invalid_rate,positive_weeks,zero_rate,drop_reason
product_sku_code,,,,,,
640007,18,16,0.888889,2,0.888889,rule1_positive_weeks<8 AND rule2_zero_rate>0.2
641491,4,2,0.500000,2,0.500000,rule1_positive_weeks<8 AND rule2_zero_rate>0.2
640905,10,3,0.300000,7,0.300000,rule1_positive_weeks<8 AND rule2_zero_rate>0.2


In [46]:
# testing
is_vol_nan = df[VOL_COL].isna()
is_vol_zero = df[VOL_COL].eq(0)
is_vol_neg = df[VOL_COL].lt(0)
is_vol_pos = df[VOL_COL].gt(0)

print(f"volume is NaN: {int(is_vol_nan.sum())}")
print(f"volume == 0:   {int(is_vol_zero.sum())}")
print(f"volume < 0:    {int(is_vol_neg.sum())}")
print(f"volume > 0:    {int(is_vol_pos.sum())}")

# we should now have some volume == 0, but no rows with volume < 0 or NaN

volume is NaN: 0
volume == 0:   2219
volume < 0:    0
volume > 0:    92777


In [47]:
# validate again
err_vol = validate_nielsen_total_volume(df)

[validate_nielsen_total_volume] error rows: 2219


##### Method 2
simply just remove all the leftover invalid volume rows

In [48]:
err_nielsen_total_volume = validate_nielsen_total_volume(df)

[validate_nielsen_total_volume] error rows: 2219


In [49]:
before_len = len(df)

bad_vol_idx = err_nielsen_total_volume.index.unique()
df = df.drop(index=bad_vol_idx)

after_len = len(df)
dropped = before_len - after_len

print(f"Before df length: {before_len}")
print(f"After df length:  {after_len}")
print(f"Rows dropped:     {dropped}")

Before df length: 94996
After df length:  92777
Rows dropped:     2219


### final validate

In [50]:
err_product_sku_code = validate_product_sku_code(df)
err_customer = validate_customer(df)
err_year = validate_year(df)
err_yearweek = validate_yearweek(df)
err_nielsen_total_volume = validate_nielsen_total_volume(df)
err_promotion_indicator = validate_promotion_indicator(df)
err_top_brand = validate_top_brand(df)
err_flavor_internal = validate_flavor_internal(df)
err_pack_type_internal = validate_pack_type_internal(df)
err_pack_size_internal = validate_pack_size_internal(df)
err_units_per_pack = validate_units_per_pack(df)
err_material_medium_description = validate_material_medium_description(df)
err_price_per_item = validate_price_per_item(df)

[validate_product_sku_code] error rows: 0
[validate_customer] error rows: 0
[validate_year] error rows: 0
[validate_yearweek] error rows: 0
[validate_nielsen_total_volume] error rows: 0
[validate_promotion_indicator] error rows: 0
[validate_top_brand] error rows: 0
[validate_flavor_internal] error rows: 0
[validate_pack_type_internal] error rows: 0
[validate_pack_size_internal] error rows: 0
[validate_units_per_pack] error rows: 0
[validate_material_medium_description] error rows: 0
[validate_price_per_item] error rows: 0


In [51]:
# save dataset
df.to_csv("../dataset/dataset_cleaned.csv", index=False)